# Data Acquisition for Electricity Price Forecasting

This notebook handles downloading and caching all required data for both DE-LU and ES zones.

## Data Sources
1. **Historical DAA Prices**: energy-charts.info
2. **Generation Data**: ENTSO-E Transparency Platform
3. **Weather Data**: Open-Meteo API / ERA5
4. **Fuel Prices**: ICE/EEX market data

## Time Range
- Training data: 2020-01-01 to 2026-05-05 (current date)
- Evaluation window: 2026-05-11 02:00 to 2026-05-12 01:00

In [13]:
# Install dependencies if needed
%pip install -q pandas numpy requests tqdm matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm

import src.data.loaders

print("Imports successful!")
print(f"Current date: {datetime.now()}")

Imports successful!
Current date: 2026-05-06 10:14:40.668345


In [ ]:
import importlib
importlib.reload(src.data.loaders)

In [15]:
from src.data.loaders import EnergyChartsLoader, WeatherDataLoader

## 1. Download Historical DAA Prices

In [ ]:
# Initialize loader
energy_loader = EnergyChartsLoader(cache_dir=Path('../data/raw'))

# Define date range
start_date = '2026-01-01'
end_date = '2026-05-03'

zones = ['DE-LU', 'ES']

# Download prices for both zones
prices_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading DAA prices for {zone}")
    print(f"{'='*60}")
    
    df = energy_loader.load_day_ahead_prices(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False  # Set to True after first run
    )
    
    prices_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"\nPrice statistics for {zone}:")
    print(df['price_eur_mwh'].describe())
    print(f"\nNegative prices: {(df['price_eur_mwh'] < 0).sum()} hours ({(df['price_eur_mwh'] < 0).mean()*100:.2f}%)")

## 2. Download Generation Data

In [ ]:
# Download generation data for both zones
generation_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading generation data for {zone}")
    print(f"{'='*60}")
    
    df = energy_loader.load_generation_data(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False
    )
    
    generation_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"\nGeneration mix statistics for {zone}:")
    print(df.describe())

## 3. Download Weather Data

In [ ]:
# Initialize weather loader
weather_loader = WeatherDataLoader(cache_dir=Path('../data/external'))

# Download weather data for both zones
weather_data = {}

for zone in zones:
    print(f"\n{'='*60}")
    print(f"Downloading weather data for {zone}")
    print(f"{'='*60}")
    
    df = weather_loader.load_weather_data(
        zone=zone,
        start_date=start_date,
        end_date=end_date,
        use_cache=False
    )
    
    weather_data[zone] = df
    
    print(f"\nLoaded {len(df):,} hourly records")
    print(f"\nWeather statistics for {zone}:")
    print(df.describe())

## 4. Data Quality Checks

In [ ]:
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

for zone in zones:
    print(f"\n{zone}:")
    print("-" * 40)
    
    # Check for missing values
    print(f"Prices - Missing values: {prices_data[zone].isnull().sum().sum()}")
    print(f"Generation - Missing values: {generation_data[zone].isnull().sum().sum()}")
    print(f"Weather - Missing values: {weather_data[zone].isnull().sum().sum()}")
    
    # Check timestamp alignment
    print(f"\nTimestamp alignment:")
    print(f"  Prices: {len(prices_data[zone])} records")
    print(f"  Generation: {len(generation_data[zone])} records")
    print(f"  Weather: {len(weather_data[zone])} records")
    
    # Check for duplicates
    print(f"\nDuplicate timestamps:")
    print(f"  Prices: {prices_data[zone]['timestamp'].duplicated().sum()}")
    print(f"  Generation: {generation_data[zone]['timestamp'].duplicated().sum()}")
    print(f"  Weather: {weather_data[zone]['timestamp'].duplicated().sum()}")